In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\Landl\Downloads\Data-Science-Projects\sec-financial-nlp


In [2]:
import pickle

# Loading saved texts
with open("data/processed/all_texts.pkl", "rb") as f:
    all_texts = pickle.load(f)

print(f"Loaded {len(all_texts)} texts from the processed data.")

Loaded 125 texts from the processed data.


In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "text-classification",
    model='ProsusAI/finbert',
    tokenizer='ProsusAI/finbert',
    device=0
  )  # Use GPU if available or set to -1 for CPU

print('FinBERT loaded')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT loaded


In [6]:
# Sentiment on all 8-k filings

import pandas as pd

# Filter only 8-K texts
eightk_keys = [k for k in all_texts if "8-K" in k]
eightk_texts = [" ".join(all_texts[k].split()[:500]) for k in eightk_keys]

print(f"Total 8-Ks to analyze: {len(eightk_texts)}")

# Run in one batched call
batch_results = sentiment_pipeline(
    eightk_texts,
    truncation=True,
    max_length=512,
    batch_size=16
)

# Built results dataframe
records = []
for key, result in zip(eightk_keys, batch_results):
    records.append({
        "key": key,
        "ticker": key.split("_")[0],
        "label": result["label"],
        "score": round(result["score"], 4)
    })

df_sentiment = pd.DataFrame(records)
print(df_sentiment.head(10))

Total 8-Ks to analyze: 100
                             key ticker    label   score
0  AAPL_8-K_0000320193-22-000069   AAPL  neutral  0.9433
1  AAPL_8-K_0000320193-22-000107   AAPL  neutral  0.9430
2  AAPL_8-K_0000320193-23-000005   AAPL  neutral  0.9431
3  AAPL_8-K_0000320193-23-000063   AAPL  neutral  0.9433
4  AAPL_8-K_0000320193-23-000075   AAPL  neutral  0.9430
5  AAPL_8-K_0000320193-23-000104   AAPL  neutral  0.9432
6  AAPL_8-K_0000320193-24-000005   AAPL  neutral  0.9432
7  AAPL_8-K_0000320193-24-000067   AAPL  neutral  0.9433
8  AAPL_8-K_0000320193-24-000080   AAPL  neutral  0.9432
9  AAPL_8-K_0000320193-24-000120   AAPL  neutral  0.9431


In [7]:
# Pick one 8-K and print what we actually extracted
sample_key = "AAPL_8-K_0000320193-22-000069"
print(all_texts[sample_key][:1000])

8-K 1 aapl-20220728.htm 8-K aapl-20220728 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM CURRENT REPORT Pursuant to Section 13 OR 15(d) of The Securities Exchange Act of 1934 Date of Report (Date of earliest event reported) (Exact name of Registrant as specified in its charter) (State or other jurisdiction of incorporation) (Commission File Number) (I.R.S. Employer Identification No.) , (Address of principal executive offices) (Zip Code) ( ) (Registrant’s telephone number, including area code) Not applicable (Former name or former address, if changed since last report.) Check the appropriate box below if the Form 8-K filing is intended to simultaneously satisfy the filing obligation of the Registrant under any of the following provisions: Written communications pursuant to Rule 425 under the Securities Act (17 CFR 230.425) Soliciting material pursuant to Rule 14a-12 under the Exchange Act (17 CFR 240.14a-12) Pre-commencement communications pursuant to Rule

In [8]:
def extract_8k_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Extract first document only
    start = content.find("<DOCUMENT>")
    end = content.find("</DOCUMENT>") + len("</DOCUMENT>")
    doc = content[start:end]
    
    # Parse and clean HTML
    soup = BeautifulSoup(doc, "html.parser")
    for tag in soup.find_all(re.compile(r'^ix:')):
        tag.decompose()
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    text = soup.get_text(separator=" ", strip=True)
    
    # 8-K specific markers — try each in order
    markers = ["Item 2.02", "Item 7.01", "Item 8.01", "Item 1.01", "ITEM 2.02", "ITEM 7.01"]
    
    for marker in markers:
        pos = text.find(marker)
        if pos != -1:
            return text[pos:]
    
    # Fallback — skip first 3000 chars (cover page) and return rest
    return text[3000:]

In [10]:
from bs4 import BeautifulSoup
import re

def extract_clean_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    start = content.find("<DOCUMENT>")
    end = content.find("</DOCUMENT>") + len("</DOCUMENT>")
    doc = content[start:end]
    
    soup = BeautifulSoup(doc, "html.parser")
    for tag in soup.find_all(re.compile(r'^ix:')):
        tag.decompose()
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    text = soup.get_text(separator=" ", strip=True)
    
    first = text.find("PART I ")
    second = text.find("PART I ", first + 1)
    if second == -1:
        second = first
    if second == -1:
        second = 0
        
    return text[second:]

In [11]:
print(all_texts_fixed["AAPL_8-K_0001193125-22-278435"][:500])

In [12]:
# Check the raw file directly, bypassing our extraction
import os

filepath = "data/raw/sec-edgar-filings/AAPL/8-K/0001193125-22-278435/full-submission.txt"
with open(filepath, "r", encoding="utf-8") as f:
    raw = f.read()

print(len(raw))
print(raw[:1000])

299395
<SEC-DOCUMENT>0001193125-22-278435.txt : 20221107
<SEC-HEADER>0001193125-22-278435.hdr.sgml : 20221107
<ACCEPTANCE-DATETIME>20221107062707
ACCESSION NUMBER:		0001193125-22-278435
CONFORMED SUBMISSION TYPE:	8-K
PUBLIC DOCUMENT COUNT:		15
CONFORMED PERIOD OF REPORT:	20221106
ITEM INFORMATION:		Regulation FD Disclosure
ITEM INFORMATION:		Financial Statements and Exhibits
FILED AS OF DATE:		20221107
DATE AS OF CHANGE:		20221107

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Apple Inc.
		CENTRAL INDEX KEY:			0000320193
		STANDARD INDUSTRIAL CLASSIFICATION:	ELECTRONIC COMPUTERS [3571]
		IRS NUMBER:				942404110
		STATE OF INCORPORATION:			CA
		FISCAL YEAR END:			0930

	FILING VALUES:
		FORM TYPE:		8-K
		SEC ACT:		1934 Act
		SEC FILE NUMBER:	001-36743
		FILM NUMBER:		221363624

	BUSINESS ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		CITY:			CUPERTINO
		STATE:			CA
		ZIP:			95014
		BUSINESS PHONE:		(408) 996-1010

	MAIL ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		CITY:			CUPERTINO
	

## Consolidated Extraction (Final Version)
Previous attempts failed because:
- extract_clean_text() used "PART I" marker which doesn't exist in 8-K filings
- 8-K specific extraction returned 0 chars for some filing formats
- PyTorch was CPU-only, upgraded to CUDA for GPU support

This cell combines both extractors with a 200-char filter for unusable filings.

In [14]:
import pickle
import os
from bs4 import BeautifulSoup
import re

# Load good data
with open("data/processed/all_texts.pkl", "rb") as f:
    all_texts_fixed = pickle.load(f)

print(f"Loaded {len(all_texts_fixed)} filings from yesterday")

# Re-extract 8-Ks only with the new function
def extract_8k_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    start = content.find("<DOCUMENT>")
    end = content.find("</DOCUMENT>") + len("</DOCUMENT>")
    doc = content[start:end]
    soup = BeautifulSoup(doc, "html.parser")
    for tag in soup.find_all(re.compile(r'^ix:')):
        tag.decompose()
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    markers = ["Item 2.02", "Item 7.01", "Item 8.01", "Item 1.01",
               "ITEM 2.02", "ITEM 7.01"]
    for marker in markers:
        pos = text.find(marker)
        if pos != -1:
            return text[pos:]
    return text[3000:]

# Only re-extract 8-Ks and replace them in the dictionary
for ticker in ["AAPL", "MSFT", "GOOGL", "JPM", "TSLA"]:
    folder = f"data/raw/sec-edgar-filings/{ticker}/8-K"
    if not os.path.exists(folder):
        continue
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".txt"):
                filepath = os.path.join(root, file)
                key = f"{ticker}_8-K_{os.path.basename(root)}"
                try:
                    text = extract_8k_text(filepath)
                    if len(text) > 200:
                        all_texts_fixed[key] = text
                    else:
                        # Remove bad extractions from dictionary
                        all_texts_fixed.pop(key, None)
                except Exception as e:
                    print(f"✗ {key} — {e}")

print(f"Total usable filings: {len(all_texts_fixed)}")

# Save updated version
with open("data/processed/all_texts.pkl", "wb") as f:
    pickle.dump(all_texts_fixed, f)

print("Saved.")

Loaded 125 filings from yesterday
Total usable filings: 115
Saved.


In [ ]:
# Find any AAPL 8-K key
aapl_8k_keys = [k for k in all_texts_fixed if "AAPL_8-K" in k]
print(aapl_8k_keys[0])
print(all_texts_fixed[aapl_8k_keys[0]][:500])

AAPL_8-K_0000320193-22-000069
Item 2.02    Results of Operations and Financial Condition. On July 28, 2022, Apple Inc. (“Apple”) issued a press release regarding Apple’s financial results for its third fiscal quarter ended June 25, 2022. A copy of Apple’s press release is attached hereto as Exhibit 99.1. The information contained in this Current Report shall not be deemed “filed” for purposes of Section 18 of the Securities Exchange Act of 1934, as amended (the “Exchange Act”), or incorporated by reference in any filing unde


In [16]:
import pandas as pd

eightk_keys = [k for k in all_texts_fixed if "8-K" in k]
eightk_texts = [" ".join(all_texts_fixed[k].split()[:500]) for k in eightk_keys]

print(f"Total 8-Ks to analyze: {len(eightk_texts)}")

batch_results = sentiment_pipeline(
    eightk_texts,
    truncation=True,
    max_length=512,
    batch_size=16
)

records = []
for key, result in zip(eightk_keys, batch_results):
    records.append({
        "key": key,
        "ticker": key.split("_")[0],
        "label": result["label"],
        "score": round(result["score"], 4)
    })

df_sentiment = pd.DataFrame(records)
print(df_sentiment.head(10))
print("\nSentiment distribution:")
print(df_sentiment["label"].value_counts())

Total 8-Ks to analyze: 90
                             key ticker    label   score
0  AAPL_8-K_0000320193-22-000069   AAPL  neutral  0.9452
1  AAPL_8-K_0000320193-22-000107   AAPL  neutral  0.9450
2  AAPL_8-K_0000320193-23-000005   AAPL  neutral  0.9451
3  AAPL_8-K_0000320193-23-000063   AAPL  neutral  0.9450
4  AAPL_8-K_0000320193-23-000075   AAPL  neutral  0.9450
5  AAPL_8-K_0000320193-23-000104   AAPL  neutral  0.9451
6  AAPL_8-K_0000320193-24-000005   AAPL  neutral  0.9454
7  AAPL_8-K_0000320193-24-000067   AAPL  neutral  0.9452
8  AAPL_8-K_0000320193-24-000080   AAPL  neutral  0.9453
9  AAPL_8-K_0000320193-24-000120   AAPL  neutral  0.9452

Sentiment distribution:
label
neutral    90
Name: count, dtype: int64


In [17]:
# Test FinBERT on a clearly positive sentence
test_positive = "Apple reported record breaking revenue of $100 billion, exceeding all analyst expectations with massive profit growth."
test_negative = "Apple reported devastating losses, missing targets badly with revenue collapsing amid severe economic headwinds."

print(sentiment_pipeline(test_positive))
print(sentiment_pipeline(test_negative))

[{'label': 'positive', 'score': 0.9417855143547058}]
[{'label': 'negative', 'score': 0.9702388048171997}]


In [18]:
# Print exactly what we're feeding into FinBERT for the first 8-K
sample_key = eightk_keys[0]
sample_text = " ".join(all_texts_fixed[sample_key].split()[:500])
print(repr(sample_text[:300]))

'Item 2.02 Results of Operations and Financial Condition. On July 28, 2022, Apple Inc. (“Apple”) issued a press release regarding Apple’s financial results for its third fiscal quarter ended June 25, 2022. A copy of Apple’s press release is attached hereto as Exhibit 99.1. The information contained i'


In [19]:
filepath = "data/raw/sec-edgar-filings/AAPL/8-K/0000320193-22-000069/full-submission.txt"
with open(filepath, "r", encoding="utf-8") as f:
    content = f.read()

types = []
for line in content.split("\n"):
    if line.startswith("<TYPE>"):
        types.append(line.replace("<TYPE>", "").strip())

print(types)

['8-K', 'EX-99.1', 'EX-101.SCH', 'EX-101.DEF', 'EX-101.LAB', 'EX-101.PRE', 'GRAPHIC', 'XML', 'XML', 'EXCEL', 'XML', 'XML', 'XML', 'JSON', 'ZIP']


### Root cause (confirmed right now)
The REAL earnings content is inside EX-99.1 (Exhibit 99.1)
not in the main 8-K document. The main document just says
"we attached the press release as Exhibit 99.1."

EX-99.1 is what contains:
- Actual revenue numbers
- CEO quotes
- Growth commentary
- Forward guidance

### we're doing right now
Confirmed the filing contains these documents:
['8-K', 'EX-99.1', 'EX-101.SCH', ...]

EX-99.1 is right there — second document in the bundle.
Next step: extract EX-99.1 instead of the main 8-K document.
This will give FinBERT real earnings content to analyze.

In [20]:
def extract_8k_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Split into individual documents
    documents = content.split("<DOCUMENT>")
    
    # Find EX-99.1 specifically — that's where real earnings content lives
    target_doc = None
    for doc in documents:
        if doc.strip().startswith("<TYPE>EX-99.1"):
            target_doc = doc
            break
    
    # Fallback to main 8-K document if no EX-99.1 found
    if target_doc is None:
        for doc in documents:
            if doc.strip().startswith("<TYPE>8-K"):
                target_doc = doc
                break
    
    if target_doc is None:
        return ""
    
    # Clean HTML
    soup = BeautifulSoup(target_doc, "html.parser")
    for tag in soup.find_all(re.compile(r'^ix:')):
        tag.decompose()
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    text = soup.get_text(separator=" ", strip=True)
    
    # Skip any remaining header junk
    return text[500:] if len(text) > 500 else text

In [21]:
sample = extract_8k_text("data/raw/sec-edgar-filings/AAPL/8-K/0000320193-22-000069/full-submission.txt")
print(len(sample))
print(sample[:1000])

10005
s per diluted share of $1.20. “This quarter’s record results speak to Apple’s constant efforts to innovate, to advance new possibilities, and to enrich the lives of our customers,” said Tim Cook, Apple’s CEO. “As always, we are leading with our values, and expressing them in everything we build, from new features that are designed to protect user privacy and security, to tools that will enhance accessibility, part of our longstanding commitment to create products for everyone.” “Our June quarter results continued to demonstrate our ability to manage our business effectively despite the challenging operating environment. We set a June quarter revenue record and our installed base of active devices reached an all-time high in every geographic segment and product category,” said Luca Maestri, Apple’s CFO. “During the quarter, we generated nearly $23 billion in operating cash flow, returned over $28 billion to our shareholders, and continued to invest in our long-term growth plans.” 

In [22]:
# Check one 8-K from each company
test_files = {
    "MSFT": "data/raw/sec-edgar-filings/MSFT/8-K",
    "TSLA": "data/raw/sec-edgar-filings/TSLA/8-K",
    "GOOGL": "data/raw/sec-edgar-filings/GOOGL/8-K",
    "JPM": "data/raw/sec-edgar-filings/JPM/8-K"
}

for ticker, folder in test_files.items():
    # Get first filing
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".txt"):
                filepath = os.path.join(root, file)
                sample = extract_8k_text(filepath)
                print(f"\n{ticker} — {len(sample)} chars")
                print(sample[:300])
                break
        break

In [23]:
import os

# Get first MSFT 8-K filepath
msft_filepath = None
for root, dirs, files in os.walk("data/raw/sec-edgar-filings/MSFT/8-K"):
    for file in files:
        if file.endswith(".txt"):
            msft_filepath = os.path.join(root, file)
            break
    if msft_filepath:
        break

print(msft_filepath)

# Check document types
with open(msft_filepath, "r", encoding="utf-8") as f:
    content = f.read()

types = []
for line in content.split("\n"):
    if line.startswith("<TYPE>"):
        types.append(line.replace("<TYPE>", "").strip())

print(types)
print(f"\nTotal docs: {len(types)}")
print(f"\nDoes EX-99.1 exist: {'EX-99.1' in types}")

data/raw/sec-edgar-filings/MSFT/8-K\0000950170-23-034400\full-submission.txt
['8-K', 'EX-99.1', 'EX-101.PRE', 'EX-101.LAB', 'EX-101.DEF', 'EX-101.SCH', 'XML', 'XML', 'EXCEL', 'XML', 'XML', 'XML', 'JSON', 'ZIP']

Total docs: 14

Does EX-99.1 exist: True


In [24]:
with open(msft_filepath, "r", encoding="utf-8") as f:
    content = f.read()

documents = content.split("<DOCUMENT>")
print(f"Total splits: {len(documents)}")

# Show start of each document
for i, doc in enumerate(documents[:5]):
    print(f"\n--- Doc {i} ---")
    print(repr(doc[:100]))

Total splits: 15

--- Doc 0 ---
'<SEC-DOCUMENT>0000950170-23-034400.txt : 20230725\n<SEC-HEADER>0000950170-23-034400.hdr.sgml : 202307'

--- Doc 1 ---
'\n<TYPE>8-K\n<SEQUENCE>1\n<FILENAME>msft-20230725.htm\n<DESCRIPTION>8-K\n<TEXT>\n<XBRL>\n<?xml version="1.0'

--- Doc 2 ---
'\n<TYPE>EX-99.1\n<SEQUENCE>2\n<FILENAME>msft-ex99_1.htm\n<DESCRIPTION>EX-99.1\n<TEXT>\n<html>\n <head>\n  <t'

--- Doc 3 ---
'\n<TYPE>EX-101.PRE\n<SEQUENCE>3\n<FILENAME>msft-20230725_pre.xml\n<DESCRIPTION>XBRL TAXONOMY EXTENSION P'

--- Doc 4 ---
'\n<TYPE>EX-101.LAB\n<SEQUENCE>4\n<FILENAME>msft-20230725_lab.xml\n<DESCRIPTION>XBRL TAXONOMY EXTENSION L'


In [25]:
documents = content.split("<DOCUMENT>")

for doc in documents:
    stripped = doc.strip()
    if stripped.startswith("<TYPE>EX-99.1"):
        print("FOUND IT")
        print(repr(stripped[:200]))
        break
else:
    print("NOT FOUND")
    # Show what each doc actually starts with
    for i, doc in enumerate(documents):
        print(f"Doc {i} starts with: {repr(doc.strip()[:30])}")

FOUND IT
'<TYPE>EX-99.1\n<SEQUENCE>2\n<FILENAME>msft-ex99_1.htm\n<DESCRIPTION>EX-99.1\n<TEXT>\n<html>\n <head>\n  <title>EX-99.1</title>\n </head>\n <body style="margin: auto!important;padding: 8px;">\n  <p style="text-i'


In [26]:
with open(msft_filepath, "r", encoding="utf-8") as f:
    content = f.read()

documents = content.split("<DOCUMENT>")

for doc in documents:
    if doc.strip().startswith("<TYPE>EX-99.1"):
        # Just count ix: tags
        ix_count = doc.count("ix:")
        total_text_len = len(doc)
        print(f"Total chars: {total_text_len}")
        print(f"ix: tag count: {ix_count}")
        break

Total chars: 1649355
ix: tag count: 0


In [27]:
from bs4 import BeautifulSoup

for doc in documents:
    if doc.strip().startswith("<TYPE>EX-99.1"):
        soup = BeautifulSoup(doc, "html.parser")
        text = soup.get_text(separator=" ", strip=True)
        print(f"Extracted text length: {len(text)}")
        print(text[:500])
        break

Extracted text length: 29530
EX-99.1 2 msft-ex99_1.htm EX-99.1 EX-99.1 Exhibit 99.1 Microsoft Cloud Strength Drives Fourth Quarter Results REDMOND, Wash. — July 25, 2023 — Microsoft Corp. today announced the following results for the quarter ended June 30, 2023, as compared to the corresponding period of last fiscal year: • Revenue was $56.2 billion and increased 8% (up 10% in constant currency) • Operating income was $24.3 billion and increased 18% (up 21% in constant currency) • Net income was $20.1 billion and increased 


## What changed and why:
Removed content.find("<DOCUMENT>") approach entirely — that was the root bug, only grabbing document #1.
Used content.split("<DOCUMENT>") instead — splits all documents, lets us find EX-99.1 specifically.
Removed marker search (Item 2.02 etc.) — those markers don't exist in press releases, they were causing the fallback to text[3000:] which was returning something for AAPL but empty for others.
text[200:] — just skip the first 200 chars of SEC header metadata (EX-99.1 2 msft-ex99_1.htm...) at the top of every exhibit, then return everything else.
Redefine this function and test on both AAPL and MSFT:

In [28]:
def extract_8k_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Split into ALL documents
    documents = content.split("<DOCUMENT>")
    
    # Try EX-99.1 first — real earnings press release
    target_doc = None
    for doc in documents:
        if doc.strip().startswith("<TYPE>EX-99.1"):
            target_doc = doc
            break
    
    # Fallback to main 8-K if no EX-99.1
    if target_doc is None:
        for doc in documents:
            if doc.strip().startswith("<TYPE>8-K"):
                target_doc = doc
                break
    
    if target_doc is None or len(target_doc) < 200:
        return ""
    
    # Clean HTML
    soup = BeautifulSoup(target_doc, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    text = soup.get_text(separator=" ", strip=True)
    
    # No marker search needed — EX-99.1 is already the right content
    # skip the first 200 chars of SEC header metadata
    return text[200:] if len(text) > 200 else text

In [29]:
aapl_sample = extract_8k_text("data/raw/sec-edgar-filings/AAPL/8-K/0000320193-22-000069/full-submission.txt")
msft_sample = extract_8k_text(msft_filepath)

print("AAPL:", len(aapl_sample))
print(aapl_sample[:300])
print("\nMSFT:", len(msft_sample))
print(msft_sample[:300])

AAPL: 10305
ll-time high for all major product categories CUPERTINO, California — July 28, 2022 — Apple ® today announced financial results for its fiscal 2022 third quarter ended June 25, 2022. The Company posted a June quarter revenue record of $83.0 billion, up 2 percent year over year, and quarterly earning

MSFT: 29330
 the quarter ended June 30, 2023, as compared to the corresponding period of last fiscal year: • Revenue was $56.2 billion and increased 8% (up 10% in constant currency) • Operating income was $24.3 billion and increased 18% (up 21% in constant currency) • Net income was $20.1 billion and increased 


In [32]:
test_companies = {
    "GOOGL": "data/raw/sec-edgar-filings/GOOGL/8-K",
    "JPM": "data/raw/sec-edgar-filings/JPM/8-K",
    "TSLA": "data/raw/sec-edgar-filings/TSLA/8-K"
}

for ticker, folder in test_companies.items():
    found = False
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".txt"):
                filepath = os.path.join(root, file)
                sample = extract_8k_text(filepath)
                print(f"\n{ticker} — {len(sample)} chars")
                print(sample[:300])

                break
    if found:
        break


GOOGL — 6119 chars
TED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 8-K CURRENT REPORT Pursuant to Section 13 or 15(d) of The Securities Exchange Act of 1934 Date of Report (Date of earliest event reported) April 18, 2023 ALPHABET INC. (Exact name of registrant as specified in its charter) Del

GOOGL — 9937 chars
2023-06-02 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 8-K CURRENT REPORT Pursuant to Section 13 or 15(d) of The Securities Exchange Act of 1934 Date of Report (Date of earliest event reported) June 2, 2023 ALPHABET INC. (Exact name of registrant as specified in its 

GOOGL — 2552 chars
 today announced that Anat Ashkenazi, currently Executive Vice President and
Chief Financial Officer at Eli Lilly and Company, will join its management team as Chief Financial Officer and Senior Vice President of Google and Alphabet, effective July 31, 2024. The company had previously announced in J

GOOGL — 4810 chars
TED STATES SECURI

In [34]:
import pickle

# Load good 10-K data
with open("data/processed/all_texts.pkl", "rb") as f:
    all_texts_fixed = pickle.load(f)

print(f"Loaded {len(all_texts_fixed)} filings from pickle")

# re-extract 8-Ks with the fixed function
for ticker in ["AAPL", "MSFT", "GOOGL", "JPM", "TSLA"]:
    folder = f"data/raw/sec-edgar-filings/{ticker}/8-K"
    if not os.path.exists(folder):
        continue
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".txt"):
                filepath = os.path.join(root, file)
                key = f"{ticker}_8-K_{os.path.basename(root)}"
                try:
                    text = extract_8k_text(filepath)
                    if len(text) > 500:
                        all_texts_fixed[key] = text
                    else:
                        all_texts_fixed.pop(key, None)
                except Exception as e:
                    print(f"✗ {key} — {e}")

print(f"Total usable filings: {len(all_texts_fixed)}")

Loaded 115 filings from pickle
Total usable filings: 125


In [36]:
sample_key = [k for k in all_texts_fixed if "AAPL_8-K" in k][0]
sample_text = " ".join(all_texts_fixed[sample_key].split()[:500])
print(sample_text[:300])
print(sentiment_pipeline(sample_text, truncation=True, max_length=512))

ll-time high for all major product categories CUPERTINO, California — July 28, 2022 — Apple ® today announced financial results for its fiscal 2022 third quarter ended June 25, 2022. The Company posted a June quarter revenue record of $83.0 billion, up 2 percent year over year, and quarterly earning
[{'label': 'neutral', 'score': 0.8847075700759888}]


In [9]:
import os
os.chdir("sec-financial-nlp")
print(os.getcwd())

c:\Users\Landl\Downloads\Data-Science-Projects\sec-financial-nlp


In [10]:
import pickle
with open("data/processed/all_texts.pkl", "rb") as f:
    all_texts = pickle.load(f)
print(f"Loaded {len(all_texts)} texts from the processed data.")

Loaded 115 texts from the processed data.


In [8]:
import torch
print(torch.cuda.is_available())

True


# changes happening now

Change — Extract CEO quote paragraph
Instead of first 400 words (which is mostly boilerplate numbers), we find the paragraph containing the CEO's direct quote — "said Tim Cook" or "said Satya Nadella" — and extract just that section. CEO quotes are deliberately crafted to communicate sentiment to investors.

In [12]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=0
)

print("FinBERT loaded on GPU.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT loaded on GPU.


CEO — talks about vision, strategy, product direction, customer experience, future outlook. Deliberately optimistic and forward-looking. "We're incredibly excited about our pipeline..." This is where positive/negative sentiment lives strongest.   
CFO — talks about specific numbers, margins, cash flow, guidance for next quarter. More measured but also more specific. "We expect revenue between $89-93 billion next quarter" — that's a forward guidance statement with real sentiment signal (confident guidance = positive, wide range = uncertainty = negative).
So both are valuable but for different reasons — CEO for strategic sentiment, CFO for financial guidance sentiment. That's why we want both patterns.

In [13]:
import re

def extract_ceo_quote(text):
    # Common patterns for CEO/CFO quote attribution
    patterns = [
        r'said\s+\w+\s+\w+,.*?(?:Chief Executive|CEO|President)',
        r'said\s+\w+\s+\w+,.*?(?:Chief Financial|CFO)',
    ]
    
    # Split into sentences
    sentences = text.replace('\n', ' ').split('.')
    
    # Find sentence containing CEO attribution
    for i, sentence in enumerate(sentences):
        for pattern in patterns:
            if re.search(pattern, sentence, re.IGNORECASE):
                # Return 3 sentences before and including the quote
                start = max(0, i-3)
                quote_block = '. '.join(sentences[start:i+1])
                if len(quote_block) > 50:
                    return quote_block
    
    # Fallback — return first 300 words
    return ' '.join(text.split()[:300])

In [15]:
sample_key = [k for k in all_texts if "AAPL_8-K" in k][0]
quote = extract_ceo_quote(all_texts[sample_key])
print(quote)

Item 2.02 Results of Operations and Financial Condition. On July 28, 2022, Apple Inc. (“Apple”) issued a press release regarding Apple’s financial results for its third fiscal quarter ended June 25, 2022. A copy of Apple’s press release is attached hereto as Exhibit 99.1. The information contained in this Current Report shall not be deemed “filed” for purposes of Section 18 of the Securities Exchange Act of 1934, as amended (the “Exchange Act”), or incorporated by reference in any filing under the Securities Act of 1933, as amended, or the Exchange Act, except as shall be expressly set forth by specific reference in such a filing. Item 9.01 Financial Statements and Exhibits. (d) Exhibits. Exhibit Number Exhibit Description 99.1 Press release issued by Apple Inc. on July 28, 2022. 104 Inline XBRL for the cover page of this Current Report on Form 8-K. SIGNATURE Pursuant to the requirements of the Securities Exchange Act of 1934, the Registrant has duly caused this report to be signed on 

In [16]:
# Check what the first AAPL 8-K actually contains
sample_key = [k for k in all_texts if "AAPL_8-K" in k][0]
print(sample_key)
print(all_texts[sample_key][:300])

AAPL_8-K_0000320193-22-000069
Item 2.02    Results of Operations and Financial Condition. On July 28, 2022, Apple Inc. (“Apple”) issued a press release regarding Apple’s financial results for its third fiscal quarter ended June 25, 2022. A copy of Apple’s press release is attached hereto as Exhibit 99.1. The information containe


In [17]:
def extract_8k_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Step 1 — split all documents
    documents = content.split("<DOCUMENT>")
    
    # Step 2 — find EX-99.1
    target_doc = None
    for doc in documents:
        if doc.strip().startswith("<TYPE>EX-99.1"):
            target_doc = doc
            break
    
    # Step 3 — fallback to main 8-K if no EX-99.1
    if target_doc is None:
        for doc in documents:
            if doc.strip().startswith("<TYPE>8-K"):
                target_doc = doc
                break
    
    # Step 4 — return empty if still nothing
    if target_doc is None or len(target_doc) < 200:
        return ""
    
    # Step 5 — clean with BeautifulSoup
    soup = BeautifulSoup(target_doc, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    for tag in soup.find_all(re.compile(r'^ix:')):
        tag.decompose()
    
    text = soup.get_text(separator=" ", strip=True)
    
    # Step 6 — skip header junk
    return text[200:] if len(text) > 200 else text

In [18]:
import os

for ticker in ["AAPL", "MSFT", "GOOGL", "JPM", "TSLA"]:
    folder = f"data/raw/sec-edgar-filings/{ticker}/8-K"
    if not os.path.exists(folder):
        continue
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".txt"):
                filepath = os.path.join(root, file)
                key = f"{ticker}_8-K_{os.path.basename(root)}"
                try:
                    text = extract_8k_text(filepath)
                    if len(text) > 500:
                        all_texts_fixed[key] = text
                    else:
                        all_texts_fixed.pop(key, None)
                except Exception as e:
                    print(f"✗ {key} — {e}")

print(f"Total filings: {len(all_texts_fixed)}")

with open("data/processed/all_texts.pkl", "wb") as f:
    pickle.dump(all_texts_fixed, f)

print("Saved.")

✗ AAPL_8-K_0000320193-22-000069 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-22-000107 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-23-000005 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-23-000063 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-23-000075 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-23-000104 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-24-000005 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-24-000067 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-24-000080 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0000320193-24-000120 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0001140361-23-011192 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0001140361-23-023909 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0001140361-24-010155 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_0001140361-24-024352 — name 'BeautifulSoup' is not defined
✗ AAPL_8-K_000114036

KeyboardInterrupt: 

new notebook this is for debuggiing go to notebook 3
